In [ ]:
import os
import re
import sys

import pandas as pd
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

credentials = os.getenv("GIGACHAT_CREDENTIALS") or os.getenv("GIGA_KEY")

llm = GigaChat(
    credentials=credentials,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Извлеки количество проживающих из текста заявки на аренду. "
            "Верни только одно целое число без слов и пояснений."
        ),
        (
            "human",
            "Текст заявки:\n{text}",
        ),
    ]
)

chain = prompt | llm | StrOutputParser()

df = pd.read_csv("rental_32.csv", sep=";")
print(df.head())
print(df.dtypes)

results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        raw = chain.invoke({"text": text}).strip()
        match = re.search(r"\\d+", raw)
        result = match.group(0) if match else "ERROR: no number"
        results.append(result)
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")

df["result_num"] = pd.to_numeric(df["result"], errors="coerce")
df["is_correct"] = df["amount"] == df["result_num"]

correct = df["is_correct"].sum()
errors = len(df) - correct
accuracy = correct / len(df)

print(df[["amount", "result", "result_num", "is_correct"]])
print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")
